# CelebA GAN Benchmark

Run every cell top-to-bottom after editing `config/run.yaml`.

**Crash recovery** - if the kernel dies mid-run, restart it and re-run every cell from the top. Each training cell automatically detects `checkpoints/latest.pt` and resumes from the last saved step. Set `FORCE_FRESH_RUN = True` in Cell 2 only when you want to start completely from scratch.


In [ ]:
# Cell 1 - imports (safe to re-run after kernel restart)
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.config import load_run_config, config_summary
from src.pipeline import (
    preflight, prepare_data, run_smoke_tests,
    run_main_comparison, run_lambda_ablation,
    build_report, summarize_artifacts,
)

In [ ]:
# Cell 2 - resume flag
# Set to True to wipe previous checkpoints and train from scratch.
# Leave False (the default) to auto-resume from a crashed or interrupted run.
FORCE_FRESH_RUN = False

## Environment and current configuration


In [ ]:
# Cell 3 - preflight checks (safe to re-run)
config = load_run_config(ROOT / "config/run.yaml", ROOT)
environment = preflight(config)
print(config_summary(config))
environment

## Build or validate the configuration-specific manifest and cache


In [ ]:
# Cell 4 - data preparation (idempotent - safe to re-run)
prepared = prepare_data(config)
print(f"Images: {len(prepared['manifest'])}; workers selected: {prepared['workers']}")
prepared['manifest'].head()

## Smoke test


In [ ]:
# Cell 5 - smoke test (always runs fresh, no resume)
smoke_results = run_smoke_tests(config, prepared, steps=2)
smoke_results

## Main comparison - DCGAN, WGAN-Clipping, WGAN-GP

> Re-running this cell after a kernel crash will automatically resume each model
> from its latest checkpoint. Set `FORCE_FRESH_RUN = True` in Cell 2 to start over.


In [ ]:
# Cell 6 - main comparison (crash-safe: re-run after kernel restart resumes automatically)
main_results = run_main_comparison(config, prepared, resume=not FORCE_FRESH_RUN)
main_results

## WGAN-GP lambda ablation

> Same crash-recovery behaviour as the main comparison cell.


In [ ]:
# Cell 7 - lambda ablation (crash-safe)
ablation_results = run_lambda_ablation(config, prepared, resume=not FORCE_FRESH_RUN)
ablation_results

## Analysis, report, and artifact summary


In [ ]:
# Cell 8 - reporting (safe to re-run at any time)
report_paths = build_report(config)
summary = summarize_artifacts(config)
print("Generated:")
for path in report_paths:
    print(path)
summary